In [ ]:
# ================================================================
# Compare 2–3 BED-like files (smoothed only, one panel per chrom)
# Plot ONLY within REGION_STR_BED, and ONLY if all files have rows
# in that region for that chromosome.
# All files are overlaid on a single panel per chromosome.
# ================================================================

import re
import os
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt

# Fix OverflowError from too many points at high DPI
mpl.rcParams["agg.path.chunksize"] = 10000

# ----------------- Shared helpers -----------------

def chrom_sort_key(ch: str):
    """
    Sort like chr1..chr22, chrX, chrY, chrM, and then MATERNAL/PATERNAL.
    """
    m = re.match(r'^chr(?P<num>\d+|X|Y|M)(?:_(?P<hap>\w+))?$', ch)
    if not m:
        return (float("inf"), 0, ch)
    num, hap = m.group("num"), (m.group("hap") or "").upper()
    if num == "X":
        n = 23
    elif num == "Y":
        n = 24
    elif num in ("M", "MT"):
        n = 25
    else:
        try:
            n = int(num)
        except ValueError:
            n = float("inf")
    hap_order = {"MATERNAL": 0, "PATERNAL": 1}.get(hap, 2)
    return (n, hap_order, ch)

# =======================
# Parameters (edit these)
# =======================
FILEA_PATH = "/private/groups/migalab/dan/10_27_25_R1041_UL_Dimelo_CENPA_1/20251027_2309_4A_PBE44270_43c879af/pod5_skip/merged_UL_DiMeLo_CENPAyoung_1_5mA_6mC_winnowmap_MD_corrected_cen_cpg_pileup_trimmed_tagged.bed"
FILEB_PATH = "/private/groups/migalab/dan/08_05_24_R1041_ULadapt_Dimelo_H3K9ME3/08_05_24_R1041_ULadapt_Dimelo_H3K4ME3/08_05_24_R1041_ULadapt_Dimelo_H3K4ME3_1/20240805_1148_1F_PAU87705_0451cc00/pod5/08_05_24_R1041_ULadapt_Dimelo_H3K4ME3_mA_mC_winnowmap_sorted_MD_modkit_cpg_pileup_active_trimmed.bed"
FILEC_PATH = None   # set a real path here if you want 3 files, else keep None

LABELA = "A iPSC (GM26105_LCL)"
LABELB = "B (LCL)"
LABELC = "C iPSC (GM27730_PBMC)"   # used only if FILEC_PATH is not None

# colors for the smoothed lines
# You can change these later to any Matplotlib color:
# - Hex: "#RRGGBB"
# - RGB tuple: (R, G, B) with values in [0,1]
COLORS = [
    "#1f77b4",  # line color for FILEA
    "#ff7f0e",  # line color for FILEB
    "#2ca02c",  # line color for FILEC (if used)
]

SUBSET_CHROMS_BED = None          # e.g., ["chr10_MATERNAL"]; or None = auto

WINDOW_BP = 10000                 # smoothing window in base pairs (rolling mean)
SMOOTH_LINEWIDTH = 1.4

# ---- square panel size ----
FIG_SIZE = 5.0                    # inches; width == height
FIG_W, FIG_H = FIG_SIZE, FIG_SIZE

TITLE_PREFIX = "Fraction vs position — "
CLIP_01 = False                   # clip fraction to [0,1] if True

LOCK_GLOBAL_Y = False             # True -> same y-limits across all chromosomes

# ---- region selection options ----
# Only plot this region; all files are subset to it first.
REGION_STR_BED = "chr12_MATERNAL:35600000-36100000"

# BED file of regions to plot (chrom, start, end). Only these intervals will be shown.
REGION_BED_PATH = None
# -----------------

# Directory to save figures
PLOT_OUTDIR = "/private/groups/migalab/dan/data_analysis/HG002_figure5"
os.makedirs(PLOT_OUTDIR, exist_ok=True)

# ----------------- BED helpers -----------------

def load_bed_cols(path: str):
    """Load cols 1,2,4 from a BED-like file.... Returns dataframe or empty df if file missing."""
    if not os.path.exists(path):
        print(f"[warning] file not found: {path}")
        return pd.DataFrame(columns=["chrom", "start", "fraction"])
    df = pd.read_csv(
        path,
        sep=r"\s+",
        header=None,
        usecols=[0, 1, 3],
        names=["chrom", "start", "fraction"],
        dtype={"chrom": str},
    )
    df["start"] = pd.to_numeric(df["start"], errors="coerce")
    df["fraction"] = pd.to_numeric(df["fraction"], errors="coerce")
    df = df.dropna(subset=["start", "fraction"])
    if CLIP_01:
        df["fraction"] = df["fraction"].clip(0, 1)
    return df

def rolling_mean_bp(x_bp: np.ndarray, y: np.ndarray, window_bp: int):
    """
    Smooth y by a rolling mean over an approximate genomic window size (bp),
    using median step size between x_bp positions to determine window length.
    """
    if len(y) < 3:
        return y.copy(), len(y), 1.0
    diffs = np.diff(x_bp)
    pos = diffs[diffs > 0]
    step = float(np.median(pos)) if pos.size else 1.0
    win_n = max(3, int(round(window_bp / max(1.0, step))))
    if win_n % 2 == 0:
        win_n += 1
    y_smooth = (
        pd.Series(y)
        .rolling(win_n, center=True, min_periods=max(1, win_n // 2))
        .mean()
        .to_numpy()
    )
    return y_smooth, win_n, step

def parse_region_string_to_dict(region_str):
    """
    Parse a region like 'chr10_MATERNAL:1,000,000-2,000,000'
    into {chrom: [(start, end)]}.
    """
    region_str = region_str.strip()
    if not region_str:
        return {}

    if ":" not in region_str:
        raise ValueError(f"Region string must look like 'chrX:start-end', got: {region_str}")
    chrom_part, range_part = region_str.split(":", 1)

    m = re.match(r"\s*([0-9_,]+)\s*-\s*([0-9_,]+)\s*", range_part)
    if not m:
        raise ValueError(f"Could not parse start-end in region string: {region_str}")
    start_str, end_str = m.group(1), m.group(2)
    start = int(re.sub(r"[,_]", "", start_str))
    end = int(re.sub(r"[,_]", "", end_str))
    if end < start:
        start, end = end, start

    return {chrom_part: [(start, end)]}

def load_regions_from_bed(bed_path):
    """
    Load regions from a BED file into {chrom: [(start, end), ...]}.
    Uses columns 0,1,2 = chrom, start, end.
    """
    if not os.path.exists(bed_path):
        print(f"[warning] region BED not found: {bed_path}")
        return {}

    bed = pd.read_csv(
        bed_path,
        sep=r"\s+",
        header=None,
        usecols=[0, 1, 2],
        names=["chrom", "start", "end"],
        dtype={"chrom": str},
    )
    bed["start"] = pd.to_numeric(bed["start"], errors="coerce")
    bed["end"] = pd.to_numeric(bed["end"], errors="coerce")
    bed = bed.dropna(subset=["start", "end"])
    bed = bed[bed["end"] > bed["start"]]

    region_dict = {}
    for chrom, g in bed.groupby("chrom", sort=False):
        region_dict.setdefault(chrom, [])
        for _, row in g.iterrows():
            region_dict[chrom].append((int(row["start"]), int(row["end"])))
    return region_dict

def build_region_dict(region_str=None, bed_path=None):
    """
    Combine single region string and optional BED into a single dict.
    """
    reg = {}
    if region_str:
        reg.update(parse_region_string_to_dict(region_str))
    if bed_path:
        bed_dict = load_regions_from_bed(bed_path)
        for chrom, intervals in bed_dict.items():
            reg.setdefault(chrom, []).extend(intervals)
    return reg

# =======================
# Load data and subset
# =======================

files = []
if FILEA_PATH:
    files.append({"path": FILEA_PATH, "label": LABELA, "color": COLORS[0]})
if FILEB_PATH:
    files.append({"path": FILEB_PATH, "label": LABELB, "color": COLORS[1]})
if FILEC_PATH:
    files.append({"path": FILEC_PATH, "label": LABELC, "color": COLORS[2]})

if len(files) < 2:
    raise ValueError("Need at least two files (FILEA_PATH, FILEB_PATH).")

regions = build_region_dict(REGION_STR_BED, REGION_BED_PATH)

perfiledata = []
for f in files:
    df = load_bed_cols(f["path"])
    if regions:
        keep = []
        for chrom, intervals in regions.items():
            sub = df[df["chrom"] == chrom]
            if sub.empty:
                continue
            for (start, end) in intervals:
                keep.append(sub[(sub["start"] >= start) & (sub["start"] <= end)])
        if keep:
            df = pd.concat(keep, ignore_index=True)
        else:
            df = df.iloc[0:0]
    perfiledata.append(df)

# Filter chromosomes that all files share (within selected regions)
all_chroms = sorted(
    set.intersection(*(set(df["chrom"].unique()) for df in perfiledata)),
    key=chrom_sort_key,
)

# Build per-file dict grouped by chrom
perfile_by_chrom = []
for f, df in zip(files, perfiledata):
    chrom_dict = {c: g.sort_values("start") for c, g in df.groupby("chrom")}
    perfile_by_chrom.append(chrom_dict)

# =======================
# Plotting — all files overlaid on one panel per chromosome
# =======================

# Optional global y-limits across all chromosomes
global_ymin, global_ymax = None, None
if LOCK_GLOBAL_Y:
    ys_all = []
    for chrom in all_chroms:
        for chrom_dict in perfile_by_chrom:
            if chrom not in chrom_dict:
                continue
            d = chrom_dict[chrom]
            ys_all.append(d["fraction"].to_numpy())
    if ys_all:
        concatenated = np.concatenate(ys_all)
        global_ymin, global_ymax = float(np.nanmin(concatenated)), float(np.nanmax(concatenated))

for chrom in all_chroms:
    # Single panel — all files overlaid
    fig, ax = plt.subplots(figsize=(FIG_W, FIG_H), constrained_layout=True)

    any_data = False
    x_arrays = []

    for f, chrom_dict in zip(files, perfile_by_chrom):
        if chrom not in chrom_dict:
            continue

        d = chrom_dict[chrom]
        if d.empty:
            continue

        x = d["start"].to_numpy()
        y = d["fraction"].to_numpy()
        ys, win_n, step = rolling_mean_bp(x, y, WINDOW_BP)
        if ys.size == 0:
            continue

        ax.plot(
            x,
            ys,
            linewidth=SMOOTH_LINEWIDTH,
            color=f["color"],
            label=f["label"],
        )
        any_data = True
        x_arrays.append(x)

    if not any_data:
        plt.close(fig)
        continue

    # y-limits
    if LOCK_GLOBAL_Y and global_ymin is not None:
        ax.set_ylim(global_ymin, global_ymax)

    xmin = min(arr.min() for arr in x_arrays)
    xmax = max(arr.max() for arr in x_arrays)
    ax.set_xlim(xmin, xmax)

    ax.set_xlabel("Genomic position (bp)")
    ax.set_ylabel("Fraction")
    ax.grid(alpha=0.2)
    ax.legend()
    fig.suptitle(f"{TITLE_PREFIX}{chrom}", y=1.02)

    out_path = os.path.join(PLOT_OUTDIR, f"compare_{chrom}_zoomed_overlay.svg")
    fig.savefig(out_path, format="svg", bbox_inches="tight")
    print(f"[saved] {out_path}")

plt.show()

In [ ]:
# CENPA bar plot 
import os
from typing import List

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


# ─────────────────────────────────────────────────────────────
#  Helpers: loading and interval filtering
# ─────────────────────────────────────────────────────────────

def load_density_csv(path: str) -> pd.DataFrame:
    df = pd.read_csv(path, sep=r"\s+", header=None, engine="python")

    if df.shape[1] == 5:
        df.columns = ["chrom", "start_raw", "end_raw", "density", "coverage"]
        df["start"] = df["start_raw"].astype(str).str.strip("[],").astype(int)
        df["end"]   = df["end_raw"].astype(str).str.strip("[],").astype(int)

    elif df.shape[1] == 4:
        df.columns = ["chrom", "interval", "density", "coverage"]
        temp = df["interval"].astype(str).str.strip("[]").str.split(",", n=1, expand=True)
        df["start"] = temp[0].astype(int)
        df["end"]   = temp[1].astype(int)

    else:
        raise ValueError(
            f"Unexpected number of columns ({df.shape[1]}) in {path}. "
            "Expected 4 or 5 whitespace-separated columns."
        )

    df["density"] = df["density"].astype(float)
    df["coverage"] = df["coverage"].astype(float)
    return df[["chrom", "start", "end", "density", "coverage"]]


def _merge_intervals(intervals: np.ndarray) -> np.ndarray:
    if len(intervals) == 0:
        return intervals
    intervals = intervals[intervals[:, 0].argsort()]
    merged = [intervals[0].tolist()]
    for s, e in intervals[1:]:
        last_s, last_e = merged[-1]
        if s <= last_e:
            if e > last_e:
                merged[-1][1] = e
        else:
            merged.append([s, e])
    return np.array(merged, dtype=int)


def filter_active_away_from_cdr(
    active_df: pd.DataFrame,
    cdr_df: pd.DataFrame,
    flank_bp: int = 50_000,
) -> pd.DataFrame:
    kept_chunks = []
    for chrom, subA in active_df.groupby("chrom"):
        subC = cdr_df[cdr_df["chrom"] == chrom]
        if subC.empty:
            kept_chunks.append(subA)
            continue

        intervals = subC[["start", "end"]].to_numpy(dtype=int)
        intervals[:, 0] = intervals[:, 0] - flank_bp
        intervals[:, 1] = intervals[:, 1] + flank_bp
        merged = _merge_intervals(intervals)

        keep_mask = []
        for _, row in subA.iterrows():
            a_s, a_e = int(row["start"]), int(row["end"])
            overlaps = False
            for s, e in merged:
                if e <= a_s: continue
                if s >= a_e: break
                overlaps = True
                break
            keep_mask.append(not overlaps)

        kept_chunks.append(subA.loc[keep_mask])

    if not kept_chunks:
        return active_df.iloc[0:0].copy()
    return pd.concat(kept_chunks, ignore_index=True)


# ─────────────────────────────────────────────────────────────
#  NEW: build per-chromosome baseline from filtered active df
# ─────────────────────────────────────────────────────────────

def compute_per_chrom_baseline(filtered_active_df: pd.DataFrame) -> dict:
    """
    Compute mean density per chromosome from the filtered active windows.

    Returns
    -------
    dict : { chrom_name -> mean_density (float) }
    """
    baseline = (
        filtered_active_df
        .groupby("chrom")["density"]
        .mean()
        .to_dict()
    )
    return baseline


# ─────────────────────────────────────────────────────────────
#  Main: compute fold changes for multiple (active, CDR) pairs
# ─────────────────────────────────────────────────────────────

def compute_cdr_fold_changes(
    active_paths: List[str],
    cdr_paths: List[str],
    labels: List[str],
    flank_bp: int = 50_000,
) -> List[pd.Series]:
    """
    For each pair of (active, CDR) files:
      1. Filter active away from CDR ±flank_bp.
      2. Compute per-chromosome background mean from filtered active windows.
      3. Compute log2 fold change for each CDR row using its chromosome's baseline.
         Falls back to global mean if a chromosome has no active windows.
    """
    if not (len(active_paths) == len(cdr_paths) == len(labels)):
        raise ValueError("active_paths, cdr_paths, and labels must have the same length")

    fold_change_series_list = []

    for act_path, cdr_path, lab in zip(active_paths, cdr_paths, labels):
        print(f"\n=== Processing pair: {lab} ===")
        print(f"  active: {act_path}")
        print(f"  cdr:    {cdr_path}")

        active_df = load_density_csv(act_path)
        cdr_df    = load_density_csv(cdr_path)

        filtered_active = filter_active_away_from_cdr(active_df, cdr_df, flank_bp=flank_bp)
        n_before = len(active_df)
        n_after  = len(filtered_active)
        print(f"  active windows: {n_before} → {n_after} after exclusion")

        if n_after == 0:
            raise ValueError(f"No active windows remain after exclusion for pair '{lab}'")

        # ── Per-chromosome baseline ────────────────────────────────────
        per_chrom_baseline = compute_per_chrom_baseline(filtered_active)
        global_fallback    = filtered_active["density"].mean()

        print(f"  Per-chromosome baselines:")
        for chrom, val in sorted(per_chrom_baseline.items()):
            print(f"    {chrom}: {val:.6g}")
        print(f"  Global fallback mean: {global_fallback:.6g}")

        # ── Compute log2 FC per CDR row using its chromosome's baseline ─
        def log2_fc_for_row(row):
            bg = per_chrom_baseline.get(row["chrom"], global_fallback)
            if bg is None or bg == 0 or np.isnan(bg):
                return np.nan
            fc = row["density"] / bg
            return np.log2(fc) if fc > 0 else np.nan

        fc = cdr_df.apply(log2_fc_for_row, axis=1)
        fc.name = lab
        fold_change_series_list.append(fc)

    return fold_change_series_list


# ─────────────────────────────────────────────────────────────
#  Plotting
# ─────────────────────────────────────────────────────────────

COLORS = ["C1", "C0"]

def plot_cdr_foldchange_boxplot(
    fold_change_series_list: List[pd.Series],
    labels: List[str],
    title: str = "CDR log2 fold-change vs active background",
    jitter: float = 0.15,
    out_svg: str | None = None,
) -> None:
    if len(fold_change_series_list) != len(labels):
        raise ValueError("fold_change_series_list and labels must have same length")

    fig, ax = plt.subplots(figsize=(max(6, len(labels) * 1.2), 6))
    data = [s.dropna().values for s in fold_change_series_list]

    print("\n=== Mean log2 fold-change per dataset ===")
    for lab, arr in zip(labels, data):
        mean_fc = float(np.mean(arr)) if len(arr) > 0 else float("nan")
        print(f"  {lab}: mean log2 fold-change = {mean_fc:.3f}")

    for idx, arr in enumerate(data, start=1):
        if len(arr) == 0:
            continue
        c  = COLORS[idx - 1]
        xs = idx + np.random.uniform(-jitter, jitter, size=len(arr))
        ax.scatter(xs, arr, color=c, alpha=0.6, s=8, zorder=2)

    bp = ax.boxplot(data, patch_artist=True, showfliers=False, whis=2, zorder=3)

    for box in bp["boxes"]:
        box.set_facecolor("none")
        box.set_edgecolor("black")
        box.set_linewidth(1.5)
        box.set_zorder(3)
    for med in bp["medians"]:
        med.set_color("black")
        med.set_linewidth(2.0)
        med.set_zorder(4)
    for whisker in bp["whiskers"]:
        whisker.set_color("black")
        whisker.set_linewidth(1.0)
        whisker.set_zorder(3)
    for cap in bp["caps"]:
        cap.set_color("black")
        cap.set_linewidth(1.0)
        cap.set_zorder(3)

    ax.axhline(0, color="k", linestyle="--", linewidth=1)
    ax.set_xticks(range(1, len(labels) + 1))
    ax.set_xticklabels(labels, rotation=30, ha="right")
    ax.set_ylabel("Log2 Fold-change (CDR density / per-chrom active background mean)")
    ax.set_title(title)

    plt.tight_layout()
    if out_svg is not None:
        fig.savefig(out_svg, format="svg")
        print(f"[saved] {out_svg}")
    plt.show()


# ─────────────────────────────────────────────────────────────
#  Example usage
# ─────────────────────────────────────────────────────────────

if __name__ == "__main__":
    active_paths = [
        "/private/groups/migalab/dan/06_11_24_R1041_UL_DiMeLo_CENPAyoung_1/20240611_1126_1H_PAW33460_814408d8/pod5/CENPA_LCL_mA_active.csv",
        "/private/groups/migalab/dan/10_27_25_R1041_UL_Dimelo_CENPA_1/20251027_2309_4A_PBE44270_43c879af/pod5_skip/CENPA_iPSC_mA_active.csv",
    ]
    cdr_paths = [
        "/private/groups/migalab/dan/06_11_24_R1041_UL_DiMeLo_CENPAyoung_1/20240611_1126_1H_PAW33460_814408d8/pod5/CENPA_LCL_mA_CDR.csv",
        "/private/groups/migalab/dan/10_27_25_R1041_UL_Dimelo_CENPA_1/20251027_2309_4A_PBE44270_43c879af/pod5_skip/CENPA_HG002_mA_CDR.csv",
    ]
    labels = [
        "LCL_CENPA_mA",
        "iPSC_CENPA_mA",
    ]

    fcs = compute_cdr_fold_changes(active_paths, cdr_paths, labels, flank_bp=50_000)
    out_svg = "/private/groups/migalab/dan/data_analysis/cenpa_active_vs_cdr_foldchange_boxplot.svg"
    plot_cdr_foldchange_boxplot(fcs, labels, jitter=0.05, out_svg=out_svg)

In [ ]:
def compute_cdr_fold_changes(
    active_paths: List[str],
    cdr_paths: List[str],
    labels: List[str],
    flank_bp: int = 50_000,
    baseline_files: List[str] | None = None,  # NEW: optional per-chromosome baselines
) -> List[pd.Series]:

    log2_fold_change_series_list = []

    for i, (act_path, cdr_path, lab) in enumerate(zip(active_paths, cdr_paths, labels)):
        print(f"\n=== Processing pair: {lab} ===")

        cdr_df = load_density_csv(cdr_path)

        # ── Per-chromosome baseline from file ──────────────────────────
        if baseline_files is not None:
            baseline_dict = load_baseline(baseline_files[i])
            print(f"  Using per-chromosome baseline from: {baseline_files[i]}")

            def get_log2_fc(row):
                bg = baseline_dict.get(row["chrom"], np.nan)
                if bg is None or bg == 0 or np.isnan(bg):
                    return np.nan
                fc = row["density"] / bg
                return np.log2(fc) if fc > 0 else np.nan

            log2_fc = cdr_df.apply(get_log2_fc, axis=1)

        # ── Global baseline computed from active file (original) ───────
        else:
            active_df = load_density_csv(act_path)
            filtered_active = filter_active_away_from_cdr(active_df, cdr_df, flank_bp=flank_bp)
            print(f"  active windows: {len(active_df)} → {len(filtered_active)} after exclusion")

            if len(filtered_active) == 0:
                raise ValueError(f"No active windows remain after exclusion for pair '{lab}'")

            bg_mean = filtered_active["density"].mean()
            print(f"  Global background mean: {bg_mean:.6g}")

            fc = cdr_df["density"] / bg_mean
            log2_fc = fc.apply(lambda x: np.log2(x) if x > 0 else np.nan)

        log2_fc.name = lab
        log2_fold_change_series_list.append(log2_fc)

    return log2_fold_change_series_list# ─────────────────────────────────────────────────────────────
#  Example usage
# ─────────────────────────────────────────────────────────────

if __name__ == "__main__":
    active_paths = [
        "/private/groups/migalab/dan/06_11_24_R1041_UL_DiMeLo_CENPAyoung_1/20240611_1126_1H_PAW33460_814408d8/pod5/CENPA_HG002_mCG_active.csv",  # LCL young
        "/private/groups/migalab/dan/10_27_25_R1041_UL_Dimelo_CENPA_1/20251027_2309_4A_PBE44270_43c879af/pod5_skip/CENPA_HG002_ipsc_mCG_H1L.csv",  # iPSC
    ]
    cdr_paths = [
        "/private/groups/migalab/dan/06_11_24_R1041_UL_DiMeLo_CENPAyoung_1/20240611_1126_1H_PAW33460_814408d8/pod5/CENPA_HG002_mCG_CDR.csv",
        "/private/groups/migalab/dan/10_27_25_R1041_UL_Dimelo_CENPA_1/20251027_2309_4A_PBE44270_43c879af/pod5_skip/CENPA_HG002_mCG_CDR.csv",
    ]
    labels = [
        "LCL_mCpG",
        "iPSC_mCpG",
    ]

    fcs = compute_cdr_fold_changes(
        active_paths, cdr_paths, labels,
        flank_bp=50_000,
        baseline_files=None,  # compute from active files on the fly
    )
    out_svg = "/private/groups/migalab/dan/data_analysis/mCpG_LCL_vs_iPSC_cdr_foldchange_boxplot.svg"
    plot_cdr_foldchange_boxplot(fcs, labels, jitter=0.05, out_svg=out_svg)